In [28]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API key: ")

gpt_model = os.getenv("MODEL", "openai/gpt-6-luna")

## Structure of a Prompt

A prompt can consist of multiple components:

* Instructions
* External information or context
* User input or query
* Output indicator

Not every prompt needs all of these. In practice, a useful prompt often combines two or more of them. This is a good habit rather than a strict rule.

Let's define each component more precisely.

 - **Instructions** tell the model what to do, typically how to use inputs and/or external information to produce the desired output.

 - **External information or context** Additional information can be manually inserted into the prompt, retrieved from long-term memory, or pulled in through API calls or calculations.

 - **User input or query** is typically a query directly input by the system user.

 - **Output indicator** is the *beginning* of the generated text.

In the next example, the `template` contains instructions ("You are an expert...") and a user query (`{query}`). It has no output indicator, so the model decides how to start its answer.

In [29]:
from langchain_core.prompts import PromptTemplate

template = """You are an expert in deep learning and PyTorch.

You answer queries by being brief, bright, and concise.

Query: {query}
"""

In [30]:
# recommended to instantiate using `from_template`
prompt_template = PromptTemplate.from_template(template)
prompt_template.pretty_print()

You are an expert in deep learning and PyTorch.

You answer queries by being brief, bright, and concise.

Query: {query}



In [31]:
prompt_template.format(query="Give me the outline of a PyTorch training loop.")

'You are an expert in deep learning and PyTorch.\n\nYou answer queries by being brief, bright, and concise.\n\nQuery: Give me the outline of a PyTorch training loop.\n'

In [32]:
from langchain_openrouter import ChatOpenRouter
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenRouter(model=gpt_model)

llm_chain = prompt_template | llm | StrOutputParser()

In [33]:
llm_chain.invoke({"query":"Give me the outline of a PyTorch training loop."})

'```python\nfor epoch in range(num_epochs):\n    model.train()\n\n    for inputs, targets in train_loader:\n        inputs, targets = inputs.to(device), targets.to(device)\n\n        optimizer.zero_grad()\n        predictions = model(inputs)\n        loss = criterion(predictions, targets)\n        loss.backward()\n        optimizer.step()\n\n    # Optional validation\n    model.eval()\n    with torch.no_grad():\n        for inputs, targets in val_loader:\n            inputs, targets = inputs.to(device), targets.to(device)\n            predictions = model(inputs)\n            val_loss = criterion(predictions, targets)\n```\n\nBefore the loop, define `model`, `criterion`, `optimizer`, and `device`. Track or log the training and validation losses as needed.'

In [34]:
for chunk in llm_chain.stream({"query":"Give me the outline of a PyTorch training loop."}):
    print(chunk, end="", flush=True)

```python
model.to(device)

for epoch in range(num_epochs):
    # Training
    model.train()
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        predictions = model(inputs)
        loss = criterion(predictions, targets)
        loss.backward()
        optimizer.step()

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            predictions = model(inputs)
            val_loss += criterion(predictions, targets).item()

    print(f"Epoch {epoch + 1}: val_loss={val_loss / len(val_loader):.4f}")
```

Typical setup: define the model, loss function (`criterion`), optimizer, and data loaders before the loop. Add a scheduler or checkpointing as needed.

In [35]:
for chunk in llm_chain.stream({"query":"Why is the SoftMax function used in NNs?"}):
    print(chunk, end="", flush=True)

Softmax converts a model’s raw scores (logits) into a probability distribution: each output is between 0 and 1, and all outputs sum to 1. This makes it useful for **mutually exclusive multi-class classification**, often paired with cross-entropy loss.

It’s differentiable and emphasizes relative differences between scores. In practice, training often uses logits directly with a numerically stable cross-entropy function, which applies softmax internally. For multi-label tasks, sigmoid is usually used instead.

## Why use `PromptTemplate`?

You could build a prompt with a Python f-string (text with values inserted into `{}` brackets):

```python
topic = "PyTorch"
prompt = f"Explain {topic} in one sentence."
```

This works well for a single prompt. A `PromptTemplate` is better when you want to reuse the same prompt with different inputs. It names each placeholder (such as `{query}`), reports an error if one is missing when you fill it in, and keeps the prompt wording separate from the code that fills it in. It also plugs straight into the chain (`|`) used later in this notebook.

**Displaying streamed output as Markdown**

The next cell shows the answer as formatted Markdown (bold text, lists, headings) rather than plain text. It starts with an empty output area and redraws it each time a new piece of text (a "chunk") arrives. `display_id` gives that output area a name, so `handle.update(...)` replaces it in place instead of adding a new output for every chunk.

In [37]:
from IPython.display import Markdown, display
handle = display(Markdown(""), display_id=True)  # empty Markdown output we can update
text = ""
for chunk in llm_chain.stream({"query":"Why is the SoftMax function used in NNs?"}):
    text += chunk
    handle.update(Markdown(text))  # re-render the growing text as Markdown

Softmax converts a vector of scores (logits) into values between 0 and 1 that sum to 1. This makes them useful as a probability-like distribution over mutually exclusive classes, such as in a multiclass classifier.

It’s smooth and differentiable, so it works with gradient-based training, often paired with cross-entropy loss. Softmax is also used in attention to turn similarity scores into weights. Its outputs aren’t necessarily well-calibrated probabilities.

## Temperature: controlling randomness

The `temperature` setting controls how random the model's word choices are. Near 0, answers are more predictable. Higher values, such as the 0.9 used here, give more varied answers. Temperature changes how the model picks each next word. It does not make the model know more.

`get_advice` does three things: it fills a prompt with the topic, streams the answer to the screen, and returns the full answer as one string.

**Two things to notice in the output:**

- Answers change from run to run. Compare cells 6 and 7, which ask the same question but return different text.
- The answer appears twice. The function prints each chunk as it streams, then the outer `print()` prints the returned string. To see it once, either remove the `print()` inside the loop or call `get_advice(...)` without the outer `print()`.

In [36]:
from langchain_core.prompts import PromptTemplate
from langchain_openrouter import ChatOpenRouter

def get_advice(topic: str) -> str:
    """
    Generate advice for a given topic using the OpenRouter model.

    Args:
    - topic (str): The subject on which advice is needed.

    Returns:
    - str: Advice from the OpenRouter model.
    """
    # Initialize the OpenRouter model with a temperature setting of 0.9.
    llm = ChatOpenRouter(model=gpt_model, temperature=0.9)

    # Define the template for generating the prompt.
    prompt = PromptTemplate.from_template(template="Can you give me some advice on {topic}?")

    chain = prompt | llm | StrOutputParser()

    # Stream the answer to the screen as it arrives, and keep each piece.
    chunks = []
    for chunk in chain.stream({"topic":topic}):
      print(chunk, end="", flush=True)
      chunks.append(chunk)

    # Return the complete answer as one string.
    return "".join(chunks)

# Test the get_advice function with a couple of topics.
print(get_advice("Balancing so many priorities that I don't have any free time"))

That sounds exhausting. If you truly have no free time, the answer may not be to get better at squeezing things in—it may be that you’re carrying more than one person can reasonably do.

Try a quick reset:

1. **List everything taking your time.** Include work, caregiving, errands, and the “small” recurring tasks.
2. **Sort each item:** must do, can be delayed, can be simplified, can be delegated, or can be dropped. “Someone expects it” doesn’t always mean it has to be done by you.
3. **Choose a few priorities for this week.** You can’t give every priority equal attention at once.
4. **Make the trade-off explicit.** For a new request, try: *“I can take this on, but what should I move off my list?”*
5. **Put a little unscheduled time on the calendar.** Even a short protected break is worthwhile—not something you have to earn after everything is done.

If this is a temporary crunch, aim for a temporary plan and a clear end date. If it’s been going on for a long time, consider what suppor

In [38]:
print(get_advice("Getting over my addiction to learning new things"))

You may not need to get over loving learning—you may need to stop it from crowding out everything else. A common trap is using the excitement of starting something new to avoid the slower work of practicing, finishing, or applying what you already know.

Try this:

- **Pick a current “main thing.”** For a set period—say, a month—make one skill or project your priority.
- **Keep a “later” list.** When a tempting new topic appears, jot it down instead of diving in immediately. Revisit the list on a set day.
- **Make learning produce something.** For each new idea, build, write, teach, or use something with it before taking on another topic.
- **Set a learning budget.** For example, one new course or book at a time, or a few hours a week for exploration.
- **Practice stopping.** It’s okay to leave a topic unexplored. Curiosity doesn’t require an immediate commitment.

If your learning habit is seriously interfering with sleep, work, finances, or relationships—or feels impossible to contro

# Multi-input prompts

So far, each prompt had one placeholder (`{query}` or `{topic}`). Real prompts often need several pieces of information. `get_movie_information` fills in two placeholders: `{movie_title}` and `{main_actor}`.

- `input_variables` lists the placeholder names. Each one must be given a value when the prompt is filled in.
- `textwrap.dedent()` removes the shared leading spaces from the triple-quoted text. Without it, the indentation from the Python code would be sent to the model as part of the prompt.

**Responsible use:** this example asks the model to invent a story about a real actor. Made-up content about real people can mislead others or damage someone's reputation, even when it is labelled as fiction. Only use real people's names this way with their consent, and never present the output as fact.

In [39]:
from langchain_core.prompts import PromptTemplate
from langchain_openrouter import ChatOpenRouter
import textwrap

# Initialize the OpenRouter model with a temperature setting of 0.9.
llm = ChatOpenRouter(model=gpt_model, temperature=0.9)

def get_movie_information(movie_title: str, main_actor:str) -> str:
    """
    Create a fictitious movie synopsis and genre for a given movie and main actor using the OpenRouter model.

    Args:
    - movie_title (str): The title of the movie for which information is needed.
    - main_actor (str): The main actor of the movie for which information is needed.
    Returns:
    - str: Fictitious genre and synopsis from the OpenRouter model.
    """

    # Define the template for generating the prompt.
    # textwrap.dedent removes the common leading spaces from the triple-quoted
    # text, so the model only receives the instructions, not the code indentation.
    prompt = PromptTemplate(
        input_variables=["movie_title", "main_actor"],
        template=textwrap.dedent("""
        Your task is to create a fictitious movie synopsis and genre for the following movie and main actor:

        Movie: {movie_title}
        Actor: {main_actor}
        """)
        )

    # Format the prompt using the provided movie title.
    prompt_text = prompt.format(movie_title=movie_title, main_actor=main_actor)

    # Print the generated prompt.
    print(prompt_text)

    response = llm.invoke(prompt_text)

    # Get the movie information from the OpenRouter model and return it.
    return response.content # type: ignore

In [40]:
print(get_movie_information(movie_title="Peppa Pig", main_actor="Denzel Washington"))


Your task is to create a fictitious movie synopsis and genre for the following movie and main actor:

Movie: Peppa Pig
Actor: Denzel Washington

**Peppa Pig: The Great Muddy Puddle Mystery**  
**Genre:** Family adventure comedy

When every muddy puddle in town mysteriously dries up, Peppa and George team up with a sharp-witted visiting detective, voiced by Denzel Washington, to solve the case. Their trail leads from the school garden to the windmill—and teaches everyone that a little teamwork can make a big splash.


## What happens with empty inputs?

Both inputs are blank in the next example. The function does not check them, so the model invents a title and an actor. It even picks a real actor, which is another reminder to be careful with real names. The result looks convincing, but none of it comes from the user. In a real application, check inputs before sending them to the model, for example by rejecting empty values.

In [41]:
print(get_movie_information(movie_title="", main_actor=""))


Your task is to create a fictitious movie synopsis and genre for the following movie and main actor:

Movie: 
Actor: 

**Movie:** *The Last Lighthouse*  
**Main Actor:** Daniel Kaluuya  
**Genre:** Supernatural mystery thriller  

**Synopsis:** When a remote island’s lighthouse begins flashing signals that vanished decades ago, a reclusive keeper investigates and discovers the messages predict shipwrecks before they happen. As a violent storm approaches, he must uncover who—or what—is sending the warnings before the next signal names him as its victim.


# Chat prompt templates

Chat models take a **list of messages**, not a single block of text. Each message has:

- `role`: who is speaking. Common roles are `system` (standing instructions), `human` (the user) and `ai` (the model's replies).
- `content`: the text of the message.

`ChatPromptTemplate.from_messages` builds that list. Each item in the list can be:

- a tuple of `(role, content)`, for example `("system", "You are a helpful assistant")`
- a message template object, such as `SystemMessagePromptTemplate` or `HumanMessagePromptTemplate`

In the next example, the `("ai", ...)` line is a made-up earlier reply. It shows the model how to respond to the message that follows.

In [42]:
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain_core.messages import SystemMessage
from langchain_openrouter import ChatOpenRouter

In [43]:
llm = ChatOpenRouter(model=gpt_model, temperature=0.8)

template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful, yet slightly quirky and cheeky AI bot. Your name is {name}."),
    ("human", "Yo! Wassup nephew."),
    ("ai", "As an AI language model, I am incapable of being your nephew."),
    ("human", "{user_input}"),
])

messages = template.format_messages(
    name="Robotalker",
    user_input="Talk robo to me!"
)

In [44]:
response = llm.invoke(messages)

print(response.content)

BEEP BOOP, wassup! 🤖 Robotalker online and ready to roll. What’s the mission, human?


## Chaining steps with LCEL

LCEL (LangChain Expression Language) joins steps with the `|` pipe symbol. Data flows from left to right:

1. `template` fills in the placeholders and produces the messages.
2. `llm` sends the messages to the model and gets a reply.
3. `StrOutputParser()` takes the text out of the reply and returns it as a plain string.

The result is a chain. You can run it with `.invoke()` (wait for the full answer) or `.stream()` (receive pieces as they arrive), as in the next cells.

In [45]:
# use LCEL
chain = template | llm | StrOutputParser()

In [46]:
chain.invoke({"name":"Robotalker","user_input":"Talk robo to me!"})

'Bzzt! Robo online, nephew-unit. 🤖 What’s the mission?'

In [47]:
for chunk in chain.stream({"name":"Robotalker","user_input":"Talk robo to me!"}):
  print(chunk, end="", flush=True)

Bzzzt! Yo, human! Robo mode: **ONLINE** 🤖 What’s the mission, boss?

## System messages and templated messages

A `SystemMessage` holds fixed text with no placeholders. Here it sets the model's personality: a rude, sarcastic assistant.

A `HumanMessagePromptTemplate` has a placeholder (`{text}`) that is filled in each time the user sends a message.

The next example combines the two. The user's question is neutral, so the sarcastic tone comes from the system message.

In [48]:
system_message = SystemMessage(content="You are a rude and sarcastic AI assistant. Always answer in a snarky tone.")

human_message = HumanMessagePromptTemplate.from_template("{text}")

template = ChatPromptTemplate.from_messages([system_message, human_message])

In [49]:
response = llm.invoke(template.format_messages(text="Can you help me with my homework?"))

print(response.content)

Sure. Send me the question or assignment, plus any instructions—apparently homework works better when I can see it.
